In [20]:
import ijson
path = '../data/repo_metadata.json'
with open(path, "rb") as f:
    first_repo = next(ijson.items(f, "item"))

first_repo

{'owner': 'freeCodeCamp',
 'name': 'freeCodeCamp',
 'stars': 426893,
 'forks': 41377,
 'watchers': 8588,
 'isFork': False,
 'isArchived': False,
 'languages': [{'name': 'TypeScript', 'size': 2843131},
  {'name': 'JavaScript', 'size': 725345},
  {'name': 'CSS', 'size': 207745},
  {'name': 'Dockerfile', 'size': 4446},
  {'name': 'HTML', 'size': 1341},
  {'name': 'Shell', 'size': 940}],
 'languageCount': 6,
 'topics': [{'name': 'learn-to-code', 'stars': 480},
  {'name': 'nonprofits', 'stars': 44},
  {'name': 'programming', 'stars': 1849},
  {'name': 'nodejs', 'stars': 85145},
  {'name': 'react', 'stars': 106415},
  {'name': 'd3', 'stars': 76},
  {'name': 'careers', 'stars': 60},
  {'name': 'education', 'stars': 684},
  {'name': 'teachers', 'stars': 39},
  {'name': 'javascript', 'stars': 181110}],
 'topicCount': 16,
 'diskUsageKb': 512574,
 'pullRequests': 41267,
 'issues': 19849,
 'description': "freeCodeCamp.org's open-source codebase and curriculum. Learn math, programming, and computer

In [8]:
first_repo.keys()

dict_keys(['owner', 'name', 'stars', 'forks', 'watchers', 'isFork', 'isArchived', 'languages', 'languageCount', 'topics', 'topicCount', 'diskUsageKb', 'pullRequests', 'issues', 'description', 'primaryLanguage', 'createdAt', 'pushedAt', 'defaultBranchCommitCount', 'license', 'assignableUserCount', 'codeOfConduct', 'forkingAllowed', 'nameWithOwner', 'parent'])

In [9]:
repo = {
    'owner' : first_repo['owner'],
    'name' : first_repo['name'],
    'stars' : first_repo['stars'],
    'forks' : first_repo['forks'],
    'watchers' : first_repo['watchers'],
    'languageCount' : first_repo['languageCount'],
    'description' : first_repo['description'],
    'primaryLanguage' : first_repo['primaryLanguage'],
    'createdAt' : first_repo['createdAt'],
    'pushedAt' : first_repo['pushedAt']
    }

In [23]:
def extract_languages(languages):
    result = []
    for language in languages:
        result.append(language['name'])
    return result

In [11]:
repo['languages'] = extract_languages(first_repo['languages'])

In [12]:
repo

{'owner': 'freeCodeCamp',
 'name': 'freeCodeCamp',
 'stars': 426893,
 'forks': 41377,
 'watchers': 8588,
 'languageCount': 6,
 'description': "freeCodeCamp.org's open-source codebase and curriculum. Learn math, programming, and computer science for free.",
 'primaryLanguage': 'TypeScript',
 'createdAt': '2014-12-24T17:49:19Z',
 'pushedAt': '2025-08-31T09:33:14Z',
 'languages': ['TypeScript',
  'JavaScript',
  'CSS',
  'Dockerfile',
  'HTML',
  'Shell']}

In [22]:
def extract_topic(topics):
    result = []
    for topic in topics:
        result.append(topic['name'])
    return result

In [14]:
repo


{'owner': 'freeCodeCamp',
 'name': 'freeCodeCamp',
 'stars': 426893,
 'forks': 41377,
 'watchers': 8588,
 'languageCount': 6,
 'description': "freeCodeCamp.org's open-source codebase and curriculum. Learn math, programming, and computer science for free.",
 'primaryLanguage': 'TypeScript',
 'createdAt': '2014-12-24T17:49:19Z',
 'pushedAt': '2025-08-31T09:33:14Z',
 'languages': ['TypeScript',
  'JavaScript',
  'CSS',
  'Dockerfile',
  'HTML',
  'Shell']}

In [15]:
repo['topics'] = extract_topic(first_repo['topics'])

In [18]:
import os
import csv

In [24]:
path = '../data/repo_metadata.json'

output_dir = '../data/processed_v2'
os.makedirs(output_dir, exist_ok=True)

batch_size = 100000
batch = []
file_number = 1

fieldnames = [
    'owner',
    'name',
    'stars',
    'forks',
    'watchers',
    'languageCount',
    'description',
    'primaryLanguage',
    'createdAt',
    'pushedAt',
    'languages',
    'topics',
    'isFork',
    'isArchived'
]

with open(path, 'rb') as f:
    repositories = ijson.items(f, 'item')

    for repo_data in repositories:

        repo = {
            'owner': repo_data.get('owner'),
            'name': repo_data.get('name'),
            'stars': repo_data.get('stars'),
            'forks': repo_data.get('forks'),
            'isFork' : repo_data.get('isFork'),
            'isArchived' : repo_data.get('isArchived'),
            'watchers': repo_data.get('watchers'),
            'languageCount': repo_data.get('languageCount'),
            'description': repo_data.get('description'),
            'primaryLanguage': repo_data.get('primaryLanguage'),
            'createdAt': repo_data.get('createdAt'),
            'pushedAt': repo_data.get('pushedAt'),
            'languages': extract_languages(
                repo_data.get('languages', [])
            ),
            'topics': extract_topic(
                repo_data.get('topics', [])
            )
        }

        batch.append(repo)

        if len(batch) == batch_size:

            output_path = os.path.join(
                output_dir,
                f'repositories_{file_number:02d}.csv'
            )

            with open(output_path, 'w', newline='', encoding='utf-8') as csv_file:
                writer = csv.DictWriter(
                    csv_file,
                    fieldnames=fieldnames
                )

                writer.writeheader()
                writer.writerows(batch)

            print(f'Created {output_path}')

            batch.clear()
            file_number += 1

    if batch:

        output_path = os.path.join(
            output_dir,
            f'repositories_{file_number:02d}.csv'
        )

        with open(output_path, 'w', newline='', encoding='utf-8') as csv_file:
            writer = csv.DictWriter(
                csv_file,
                fieldnames=fieldnames
            )

            writer.writeheader()
            writer.writerows(batch)

        print(f'Created {output_path}')

print('Finished!')

Created ../data/processed_v2\repositories_01.csv
Created ../data/processed_v2\repositories_02.csv
Created ../data/processed_v2\repositories_03.csv
Created ../data/processed_v2\repositories_04.csv
Created ../data/processed_v2\repositories_05.csv
Created ../data/processed_v2\repositories_06.csv
Created ../data/processed_v2\repositories_07.csv
Created ../data/processed_v2\repositories_08.csv
Created ../data/processed_v2\repositories_09.csv
Created ../data/processed_v2\repositories_10.csv
Created ../data/processed_v2\repositories_11.csv
Created ../data/processed_v2\repositories_12.csv
Created ../data/processed_v2\repositories_13.csv
Created ../data/processed_v2\repositories_14.csv
Created ../data/processed_v2\repositories_15.csv
Created ../data/processed_v2\repositories_16.csv
Created ../data/processed_v2\repositories_17.csv
Created ../data/processed_v2\repositories_18.csv
Created ../data/processed_v2\repositories_19.csv
Created ../data/processed_v2\repositories_20.csv
Created ../data/proc

In [3]:
import pandas as pd

df = pd.read_csv('../data/processed/repositories_001.csv')
df.head()

,owner,name,stars,forks,watchers,languageCount,description,primaryLanguage,createdAt,pushedAt,languages,topics
0,freeCodeCamp,freeCodeCamp,426893,41377,8588,6,freeCodeCamp.org's open-source codebase and cu...,TypeScript,2014-12-24T17:49:19Z,2025-08-31T09:33:14Z,"['TypeScript', 'JavaScript', 'CSS', 'Dockerfil...","['learn-to-code', 'nonprofits', 'programming',..."
1,codecrafters-io,build-your-own-x,415801,38976,6247,1,Master programming by recreating your favorite...,Markdown,2018-05-09T12:03:18Z,2025-08-29T00:08:20Z,['Markdown'],"['programming', 'tutorials', 'tutorial-code', ..."
2,sindresorhus,awesome,396445,31414,8011,0,😎 Awesome lists about all kinds of interesting...,NaN,2014-07-11T13:42:37Z,2025-07-18T18:37:33Z,[],"['awesome', 'awesome-list', 'unicorns', 'lists..."
3,EbookFoundation,free-programming-books,366964,64025,9879,2,:books: Freely available programming books,Python,2013-10-11T06:50:37Z,2025-08-25T21:06:44Z,"['Python', 'HTML']","['education', 'books', 'list', 'resource', 'ha..."
4,public-apis,public-apis,363505,38166,4360,2,A collective list of free APIs,Python,2016-03-20T23:49:42Z,2025-05-20T15:56:34Z,"['Python', 'Shell']","['api', 'public-apis', 'free', 'apis', 'list',..."


In [7]:
df.shape

(50000, 12)

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   owner            50000 non-null  str  
 1   name             49998 non-null  str  
 2   stars            50000 non-null  int64
 3   forks            50000 non-null  int64
 4   watchers         50000 non-null  int64
 5   languageCount    50000 non-null  int64
 6   description      48980 non-null  str  
 7   primaryLanguage  45769 non-null  str  
 8   createdAt        50000 non-null  str  
 9   pushedAt         50000 non-null  str  
 10  languages        50000 non-null  str  
 11  topics           50000 non-null  str  
dtypes: int64(4), str(8)
memory usage: 4.6 MB


In [10]:
(df['languages'] == '[]').sum()

np.int64(4241)

In [12]:
(df['topics'] == '[]').sum()

np.int64(16836)

In [13]:
((df['languages'] == '[]') & (df['topics'] == '[]')).sum()

np.int64(1800)

In [14]:
(
    df['description'].fillna('').str.strip().eq('')
    & df['languages'].eq('[]')
    & df['topics'].eq('[]')
).sum()

np.int64(152)

In [6]:
df['stars'].describe()

count     50000.000000
mean       4719.690480
std       10087.858219
min         876.000000
25%        1508.000000
50%        2261.000000
75%        4300.250000
max      426893.000000
Name: stars, dtype: float64

In [14]:
profile_repos =df[df['owner'].str.lower() == df['name'].str.lower()]

In [15]:
profile_repos.shape

(3859, 12)

In [25]:
import pandas as pd
df = pd.read_csv('../data/processed_v2/repositories_01.csv')
df.head()

,owner,name,stars,forks,watchers,languageCount,description,primaryLanguage,createdAt,pushedAt,languages,topics,isFork,isArchived
0,freeCodeCamp,freeCodeCamp,426893,41377,8588,6,freeCodeCamp.org's open-source codebase and cu...,TypeScript,2014-12-24T17:49:19Z,2025-08-31T09:33:14Z,"['TypeScript', 'JavaScript', 'CSS', 'Dockerfil...","['learn-to-code', 'nonprofits', 'programming',...",False,False
1,codecrafters-io,build-your-own-x,415801,38976,6247,1,Master programming by recreating your favorite...,Markdown,2018-05-09T12:03:18Z,2025-08-29T00:08:20Z,['Markdown'],"['programming', 'tutorials', 'tutorial-code', ...",False,False
2,sindresorhus,awesome,396445,31414,8011,0,😎 Awesome lists about all kinds of interesting...,NaN,2014-07-11T13:42:37Z,2025-07-18T18:37:33Z,[],"['awesome', 'awesome-list', 'unicorns', 'lists...",False,False
3,EbookFoundation,free-programming-books,366964,64025,9879,2,:books: Freely available programming books,Python,2013-10-11T06:50:37Z,2025-08-25T21:06:44Z,"['Python', 'HTML']","['education', 'books', 'list', 'resource', 'ha...",False,False
4,public-apis,public-apis,363505,38166,4360,2,A collective list of free APIs,Python,2016-03-20T23:49:42Z,2025-05-20T15:56:34Z,"['Python', 'Shell']","['api', 'public-apis', 'free', 'apis', 'list',...",False,False


In [28]:
df.shape

(100000, 14)

In [30]:
df['isFork'].value_counts()

isFork
False    100000
Name: count, dtype: int64

In [31]:
df['isArchived'].value_counts()

isArchived
False    92459
True      7541
Name: count, dtype: int64

In [37]:
df['pushedAt'].head()

0   2025-08-31 09:33:14+00:00
1   2025-08-29 00:08:20+00:00
2   2025-07-18 18:37:33+00:00
3   2025-08-25 21:06:44+00:00
4   2025-05-20 15:56:34+00:00
Name: pushedAt, dtype: datetime64[us, UTC]

In [44]:
df['createdAt'].head()

0    2014-12-24T17:49:19Z
1    2018-05-09T12:03:18Z
2    2014-07-11T13:42:37Z
3    2013-10-11T06:50:37Z
4    2016-03-20T23:49:42Z
Name: createdAt, dtype: str

In [36]:
df['pushedAt'] = pd.to_datetime(df['pushedAt'], utc=True)

In [47]:
df['createAt'] = pd.to_datetime(df['createdAt'], utc=True)

In [38]:
df['pushedAt'].min()

Timestamp('2010-01-23 23:20:32+0000', tz='UTC')

In [39]:
df['pushedAt'].max()

Timestamp('2025-09-01 09:39:12+0000', tz='UTC')

In [51]:
df['repo_age_days'] = (
    df['pushedAt'] - df['createdAt']
).dt.days

In [49]:
df['createdAt'] = pd.to_datetime(df['createdAt'], utc=True)

In [50]:
df[['createdAt', 'pushedAt']].dtypes

createdAt    datetime64[us, UTC]
pushedAt     datetime64[us, UTC]
dtype: object

In [63]:
(df['repo_age_days'] < 0).sum()

np.int64(0)

In [62]:
df = df[df['repo_age_days'] >= 0]


In [ ]:
df